# Search database inspection

Read-only browser for the run/task trace database (`search.db`). Run all cells from top to bottom; every cell is read-only.

In [11]:
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Locate the repository regardless of the notebook launch directory."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "search").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(f"Could not find project root from {start}")


# When launched from src/search/script this matches the module's existing
# notebook bootstrap (two directories up); parent discovery also supports Run All
# commands started at the repository root.
PROJECT_ROOT = find_project_root(Path(os.getcwd()).resolve())
TRACE_DB_PATH = PROJECT_ROOT / "search.db"

for path in (TRACE_DB_PATH,):
    if not path.is_file():
        raise FileNotFoundError(f"Database not found: {path}")

# mode=ro makes SQLite reject all write operations from this notebook.
TRACE_CONN = sqlite3.connect(f"file:{TRACE_DB_PATH}?mode=ro", uri=True)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", None)

print(f"Project root: {PROJECT_ROOT}")
print(f"Trace DB (read-only): {TRACE_DB_PATH}")


def quote_identifier(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'


def table_names(conn: sqlite3.Connection) -> list[str]:
    return [
        row[0]
        for row in conn.execute(
            "SELECT name FROM sqlite_master "
            "WHERE type='table' AND name != 'sqlite_sequence' ORDER BY name"
        )
    ]


def show_query(label: str, query: str, conn: sqlite3.Connection, *, head: int | None = None) -> pd.DataFrame:
    df = pd.read_sql_query(query, conn)
    print(f"{label}: {df.shape[0]} rows × {df.shape[1]} columns")
    display(df.head(head) if head is not None else df)
    return df

Project root: /Users/kumo/programming/competitor_product_search
Trace DB (read-only): /Users/kumo/programming/competitor_product_search/search.db


## Summary — row counts

In [12]:
row_counts = []
for database, conn in [("search.db", TRACE_CONN)]:
    for table in table_names(conn):
        row_count = conn.execute(f"SELECT COUNT(*) FROM {quote_identifier(table)}").fetchone()[0]
        row_counts.append({"database": database, "table": table, "row_count": row_count})

summary_df = pd.DataFrame(row_counts).sort_values(["database", "table"]).reset_index(drop=True)
display(summary_df)

,database,table,row_count
0,search.db,attempts,3
1,search.db,candidates,39
2,search.db,llm_calls,2
3,search.db,meta,1
4,search.db,node_events,13
5,search.db,runs,3
6,search.db,tasks,3


## search.db — runs

In [ ]:
runs_df = show_query("runs", "SELECT * FROM runs", TRACE_CONN)

## search.db — tasks

In [ ]:
product_name = "Bonkers Zoomers BBQ Beef Flavour 85g"

In [26]:
query = """
select final_provider, count(run_id) as run_count
from tasks 
group by final_provider
"""

tasks_df = show_query("tasks", query, TRACE_CONN)

tasks: 3 rows × 2 columns


,final_provider,run_count
0,NaN,16
1,duckduckgo,62
2,serper,268


In [25]:
tasks_df = show_query("tasks", f"SELECT * FROM tasks WHERE product_name like '%{product_name}%'", TRACE_CONN)

tasks: 4 rows × 24 columns


,task_id,run_id,row_index,product_name,product_key,brand_input,website,country,status,verdict,failure_kind,matched_url,matched_title,reason,layer_trace,candidates_considered,final_provider,attempt_count,error_type,error_message,traceback,started_at,finished_at,duration_ms
0,70,1b82f20e73ee4c4e8a7b8e388a0aed72,48,Bonkers Zoomers BBQ Beef Flavour 85g,6cdd29313d5675f857ee3a8bee65e233,None,tesco,uk,ok,no_match,llm_no_match,NaN,NaN,The candidate is a different product line (Dry BBQ Steak Cuts) rather than the Zoomers BBQ Beef Flavour. (via serper),"{""domain"": ""pass"", ""brand"": ""pass"", ""numeric"": ""unknown"", ""distinguishing"": ""fail""}",9,serper,2,None,None,None,2026-08-12T23:28:23.917820+00:00,2026-08-12T23:28:48.651184+00:00,24733
1,154,b5ed397fba7b407793009f18e6e1d210,48,Bonkers Zoomers BBQ Beef Flavour 85g,6cdd29313d5675f857ee3a8bee65e233,None,tesco,uk,ok,no_match,llm_no_match,NaN,NaN,Neither candidate is the Bonkers Zoomers BBQ Beef Flavour 85g; they are different Bonkers BBQ 85g product lines. (via serper),"{""domain"": ""pass"", ""brand"": ""pass"", ""numeric"": ""unknown"", ""distinguishing"": ""fail""}",10,serper,2,None,None,None,2026-08-13T03:26:16.488507+00:00,2026-08-13T03:26:36.141161+00:00,19652
2,258,34ee7c8e8cb145c9964f865ab308ec3d,48,Bonkers Zoomers BBQ Beef Flavour 85g,6cdd29313d5675f857ee3a8bee65e233,None,tesco,uk,ok,match,matched,https://www.tesco.com/shop/en-GB/products/321344321,Bonkers Dog Treats Dry BBQ Steak Cuts 85 G - Tesco Groceries,"Same flavour (BBQ), weight (85g), and brand (Bonkers BBQ) as query. (via serper)","{""domain"": ""pass"", ""brand"": ""pass"", ""numeric"": ""pass"", ""distinguishing"": ""pass""}",10,serper,2,None,None,None,2026-08-13T20:53:41.714992+00:00,2026-08-13T20:53:58.099267+00:00,16384
3,343,70030d2a35b343698e36152aff5bb04f,48,Bonkers Zoomers BBQ Beef Flavour 85g,6cdd29313d5675f857ee3a8bee65e233,None,tesco,uk,ok,match,matched,https://www.tesco.com/shop/en-GB/products/321344321,Bonkers Dog Treats Dry BBQ Steak Cuts 85 G,"Same brand, BBQ beef flavour, and 85g weight, with only a minor title variation not indicating a different SKU. (via serper)","{""domain"": ""pass"", ""brand"": ""pass"", ""numeric"": ""pass"", ""distinguishing"": ""pass""}",9,serper,2,None,None,None,2026-08-14T19:41:54.078198+00:00,2026-08-14T19:42:17.811117+00:00,23732


## search.db — attempts

In [ ]:
attempts_df = show_query("attempts", "SELECT * FROM attempts", TRACE_CONN)

## search.db — node_events

In [13]:
node_events_df = show_query("node_events", "SELECT * FROM node_events", TRACE_CONN)

node_events: 13 rows × 15 columns


,event_id,attempt_id,task_id,run_id,seq,node,status,error_kind,error_message,traceback,detail,candidates_in,candidates_out,started_at,duration_ms
0,1,1,1,3c3103868ccf44a9b0a168cee28777d9,1,search,ok,NaN,NaN,NaN,{},0,20,2026-08-17T20:15:43.606146+00:00,2818
1,2,1,1,3c3103868ccf44a9b0a168cee28777d9,2,domain_filter,ok,NaN,NaN,NaN,"{""domain_rejects"": {""host"": 10, ""not_product_page"": 5}}",20,5,2026-08-17T20:15:46.425881+00:00,0
2,3,1,1,3c3103868ccf44a9b0a168cee28777d9,3,base_match,ok,NaN,NaN,NaN,{},5,3,2026-08-17T20:15:46.426979+00:00,2281
3,4,1,1,3c3103868ccf44a9b0a168cee28777d9,4,distinguishing,ok,NaN,NaN,NaN,{},3,3,2026-08-17T20:15:48.709700+00:00,4676
4,5,1,1,3c3103868ccf44a9b0a168cee28777d9,5,aggregate,ok,NaN,NaN,NaN,{},3,3,2026-08-17T20:15:53.388016+00:00,0
5,6,2,2,2df4c9c51b9144bbbf4025d94c34bb16,1,search,ok,NaN,NaN,NaN,{},0,20,2026-08-17T20:36:52.450712+00:00,1986
6,7,2,2,2df4c9c51b9144bbbf4025d94c34bb16,2,domain_filter,ok,NaN,NaN,NaN,"{""domain_rejects"": {""host"": 10, ""not_product_page"": 5}}",20,5,2026-08-17T20:36:54.438847+00:00,0
7,8,2,2,2df4c9c51b9144bbbf4025d94c34bb16,3,base_match,error,OperationalError,unable to open database file,"Traceback (most recent call last):\n File ""/Users/kumo/programming/competitor_product_search/src/search/graph.py"", line 43, in wrapped\n out = await fn(...",{},5,0,2026-08-17T20:36:54.440152+00:00,7
8,9,3,3,82fc24a67cc143d3bb47495d66b00fdc,1,search,ok,NaN,NaN,NaN,{},0,19,2026-08-17T20:37:13.971312+00:00,3226
9,10,3,3,82fc24a67cc143d3bb47495d66b00fdc,2,domain_filter,ok,NaN,NaN,NaN,"{""domain_rejects"": {""host"": 9, ""not_product_page"": 8}}",19,2,2026-08-17T20:37:17.200862+00:00,0


## search.db — candidates

In [ ]:
candidates_df = show_query("candidates", "SELECT * FROM candidates", TRACE_CONN)

## search.db — llm_calls

In [ ]:
llm_calls_df = show_query("llm_calls", "SELECT * FROM llm_calls", TRACE_CONN)

## search.db — meta

In [ ]:
meta_df = show_query("meta", "SELECT * FROM meta", TRACE_CONN)

## search.db — views

### v_errors

In [ ]:
v_errors_df = show_query("v_errors", "SELECT * FROM v_errors", TRACE_CONN)

### v_task_result

In [ ]:
v_task_result_df = show_query("v_task_result", "SELECT * FROM v_task_result", TRACE_CONN)

### v_funnel

In [ ]:
v_funnel_df = show_query("v_funnel", "SELECT * FROM v_funnel", TRACE_CONN)

### v_run_summary

In [ ]:
v_run_summary_df = show_query("v_run_summary", "SELECT * FROM v_run_summary", TRACE_CONN)

## Close connections

The cells above use read-only connections. Run the final cell when you are finished with the notebook.

In [ ]:
TRACE_CONN.close()
print("Read-only database connections closed.")